# Chapter 4 · Rules and relationships

**PRT2030 · Dataset inspection study guide · First exposure to data analysis**

**Learning outcomes:** Apply simple size, year and date rules; explain prerequisites; retain both flagged and untested evidence; and propose investigations without guessing corrections.

**Checks in this chapter:** CHK11 Positive size; CHK12 Whole year in period; CHK13 Date order; CHK14 Year agreement.

**Suggested teaching time:** 105–120 minutes. Include the prediction questions and student activities.
Each block is a small lesson: question, rule or expectation, example, code, meaning and next action.

**How to run:** Open the extracted study-guide folder in VS Code. Run this notebook from the top.
Each notebook loads its own `Data/wildfires.csv`. Chapters 4 and 5 also load the saved results named
beside their loading code. Restarting a kernel does not delete these saved files.

**During inspection:** Keep `raw`, the imported source table, unchanged. Temporary comparisons and
converted values belong in separate objects. A finding is a reason to investigate, not an instruction
to delete a record. All full-data counts come from your file; small examples are labelled demonstrations.


## Block 1 · Load the source and the interpreted values

**Objective:** Use Chapter 3 results without repeating its original-value conversions or checks.

**Expectation:** Saved converted tables have one source-labelled row for every row in the unchanged source.

**Where it comes from:** Our explicit file workflow. Run Chapter 3 first; these saved tables replace hidden cross-notebook variables.

**Objects used here:** `raw` is the source. `number_values` and `date_values` will be the saved result tables named below, not cleaned replacements for raw.

**Method reference:** [read_csv()](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) · [to_datetime()](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html) · [len()](https://docs.python.org/3/library/functions.html#len) · [head()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html)


In [ ]:
import pandas as pd
from pathlib import Path
from inspection_helpers import summarise_check, save_check

# Start from the same unchanged teaching file in every notebook.
# Paths are relative to the study-guide folder opened in VS Code.
DATASET_PATH = Path("Data/wildfires.csv")
raw = pd.read_csv(DATASET_PATH, low_memory=False)

# An empty table cannot show that values passed a check.
if len(raw) == 0:
    raise ValueError("No records were loaded. Check Data/wildfires.csv before continuing.")

print("Source file:", DATASET_PATH)
print("Imported records:", len(raw))
print("Columns:", raw.columns.tolist())


In [ ]:
# Load numerical results created in Chapter 3, Block 4. source_row restores the
# same row labels as raw so original records and interpreted values align.
number_values = pd.read_csv("Outputs/number_values.csv", index_col="source_row")
# Date CSVs store text. Restoring the known date type is part of loading saved
# results; the original date strings were already inspected in Chapter 3.
date_values = pd.read_csv("Outputs/date_values.csv", index_col="source_row")
date_values["discovery_date_value"] = pd.to_datetime(date_values["discovery_date_value"])
date_values["contained_date_value"] = pd.to_datetime(date_values["contained_date_value"])

# This simple prerequisite catches missing rows. Equal counts alone do not prove
# unchanged contents: continue using the same source, and rerun Chapter 3 if it changes.
if len(number_values) != len(raw) or len(date_values) != len(raw):
    raise ValueError("Saved row counts differ from the source. Rerun Chapter 3 for this dataset.")

display(number_values.head(3))
display(date_values.head(3))


**Reporting reminder:** `summarise_check(ID, title, total, flagged, untested=0)` returns a
count dictionary. `save_check(check, evidence, finding, effect, next_action)` saves it and
the original supporting records and a short preview.

**Inputs beside each check:** We name the required table columns again where they are used.
Year meaning and year agreement are taught together in Block 4, so their prerequisite stays
beside its creation and use.


## Block 2 · Predict meaning-check outcomes

**Objective:** Apply a stated expectation before reading the implementation.

| Check | Meets the rule | Flagged | Untested |
| --- | --- | --- | --- |
| Positive recorded area | `0.1` acres | `0` or `-2` acres | No ordinary number is available. |
| Whole year in this extract | `1998` | `1998.5` or `2020` | No ordinary number is available. |
| Containment on/after discovery | 10 July → 12 July, or the same day | 10 July → 9 July | Either interpreted date is unavailable. |
| Reported year agrees with discovery | `2005` and discovery in 2005 | `2005` and discovery in 2004 | The reported year fails its meaning prerequisite, or the discovery date is unavailable. |

A source value can be Flagged by readability and Untested by meaning. For example, `unknown`
cannot supply a size number: it is Flagged by CHK07, then Untested by CHK11. A readable `-2`
passes CHK07 but is Flagged by CHK11. These are different questions, not conflicting decisions.

**Student activity:** Explain the outcomes for zero size, a missing containment date and a
fractional reported year. **Suggested reasoning:** Zero can be read but fails positive area;
missing containment prevents date order; a fractional year can be read but fails a whole-year rule.


## Block 3 · CHK11 · Check positive recorded area

**Objective:** Identify non-positive sizes among records that supply ordinary numbers.

**Expectation:** An ordinary size number is greater than zero. A small positive area meets the rule; zero and negative area need review.

**Where it comes from:** A classroom plausibility rule for the source area-in-acres field. Review the measurement and source before deciding on treatment.

**Objects used here:** `raw` contains original values. `number_values["fire_size_value"]` contains interpreted numbers saved by Chapter 3; unavailable results stay missing.

**Method reference:** [notna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.notna.html) · [fillna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.fillna.html) · [loc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html) · [copy()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html) · [sort_values()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html)


In [ ]:
sizes_for_rule = number_values["fire_size_value"]
size_can_be_checked = sizes_for_rule.notna()
non_positive_size = (size_can_be_checked & (sizes_for_rule <= 0)).fillna(False)
size_meaning_untested = ~size_can_be_checked
check_positive_size = summarise_check(
    "CHK11", "Positive fire size", len(raw),
    non_positive_size.sum(), size_meaning_untested.sum()
)

# The same conditions supply both the count and complete concern evidence.
evidence_positive_size = raw.loc[
    non_positive_size | size_meaning_untested, ["fire_name", "fire_size"]
].copy()
evidence_positive_size["Interpreted size"] = sizes_for_rule.loc[evidence_positive_size.index]
evidence_positive_size["Record outcome"] = "Flagged"
evidence_positive_size.loc[size_meaning_untested.loc[evidence_positive_size.index], "Record outcome"] = "Untested"
display(pd.DataFrame([check_positive_size]))
display(evidence_positive_size.head(5))
save_check(
    check_positive_size, evidence_positive_size,
    finding="Non-positive or unavailable size numbers require a decision before area analysis.",
    effect="Q2 needs usable positive sizes under the agreed case-study rule.",
    next_action="Review original measurements and unreadable inputs; agree a correction or an explicitly described analysis subset."
)


**Read the result:** Only ordinary numbers can enter positive-size comparison. Missing converted values are Untested even when their original text was Flagged by readability. An empty evidence table means this rule found no concerns, not that every measurement is accurate.



**Pause and explain:** Does passing the positive-size rule establish that 606,945 acres was measured correctly?

<details>
<summary>Suggested answer — open after discussing</summary>

No. It meets this rule, but correctness still depends on the source and measurement.

</details>


In [ ]:
# Optional context within this same size block: inspect large positive sizes.
# Being large is a reason to review context; it does not establish an error.
# sizes_for_rule is the interpreted size Series created immediately above.
positive_size_context = raw[["fire_name", "state", "fire_size"]].copy()
positive_size_context["Interpreted size"] = sizes_for_rule
positive_size_rows = (sizes_for_rule > 0).fillna(False)
largest_size_examples = positive_size_context.loc[positive_size_rows].sort_values(
    "Interpreted size", ascending=False
).head(5)
display(largest_size_examples)


**Read the examples:** These are the five largest usable positive sizes in this file. Sorting
does not determine whether they are wrong. Large real fires are possible; unusual values need
source context before any treatment is proposed.


## Block 4 · CHK12 and CHK14 · Keep year meaning beside year agreement

**Objective:** First check whether a reported year is suitable, then compare that valid year with discovery year.

**Expectation:** A year is whole and in 1992–2015. Only that valid reported year and a usable discovery date can enter year agreement.

**Where it comes from:** The source edition covers 1992–2015; a calendar year is whole. The original FIRE_YEAR field denotes the year of discovery.

**Objects used here:** `number_values["fire_year_value"]` supplies ordinary year numbers. `date_values["discovery_date_value"]` supplies interpreted discovery dates. Both tables were explicitly loaded in Block 1.

**Method reference:** [notna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.notna.html) · [between()](https://pandas.pydata.org/docs/reference/api/pandas.Series.between.html) · [fillna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.fillna.html) · [dt.year](https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.year.html) · [loc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html) · [copy()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html)


In [ ]:
reported_years = number_values["fire_year_value"]
year_has_number = reported_years.notna()
# A whole year has zero fractional remainder: 1998 % 1 is 0; 1998.5 % 1 is 0.5.
whole_reported_year = (reported_years % 1 == 0).fillna(False)
year_in_extract = reported_years.between(1992, 2015).fillna(False)
valid_reported_year = year_has_number & whole_reported_year & year_in_extract

# Present ordinary numbers can be tested. An unsuitable number is Flagged;
# an unavailable number is Untested for this meaning check.
year_meaning_flag = year_has_number & ~valid_reported_year
year_meaning_untested = ~year_has_number
check_year_meaning = summarise_check(
    "CHK12", "Whole year in the extract period", len(raw),
    year_meaning_flag.sum(), year_meaning_untested.sum()
)
evidence_year_meaning = raw.loc[
    year_meaning_flag | year_meaning_untested, ["fire_name", "fire_year"]
].copy()
evidence_year_meaning["Interpreted year"] = reported_years.loc[evidence_year_meaning.index]
evidence_year_meaning["Record outcome"] = "Flagged"
evidence_year_meaning.loc[year_meaning_untested.loc[evidence_year_meaning.index], "Record outcome"] = "Untested"
display(pd.DataFrame([check_year_meaning]))
display(evidence_year_meaning.head(5))
save_check(
    check_year_meaning, evidence_year_meaning,
    finding="Reported-year meaning or availability needs attention for this source period.",
    effect="Q1 needs whole years in the documented extract period to form appropriate annual groups.",
    next_action="Review unsuitable years and unavailable inputs using source records before choosing a year policy."
)


**Read the result:** Whole and in-period are separate decisions joined with & because both are required. valid_reported_year is True only when the numerical prerequisite and both meaning rules are met.



**Pause and explain:** Why does 2020 fail here even though it is a real calendar year?

<details>
<summary>Suggested answer — open after discussing</summary>

This check concerns the selected 1992–2015 extract. It is not a claim that 2020 is impossible in other datasets.

</details>


**Continue this same year block:** `reported_years` contains interpreted years and
`valid_reported_year` marks the whole, in-period values just checked above. The next cell uses
those results immediately. `discovery_dates` is introduced here from the saved date table.


In [ ]:
discovery_dates = date_values["discovery_date_value"]
discovery_years = discovery_dates.dt.year
# Both sides must be usable before agreement can be checked.
year_pair_can_be_checked = valid_reported_year & discovery_dates.notna()
different_years = (reported_years != discovery_years).fillna(False)
year_agreement_flag = year_pair_can_be_checked & different_years
year_agreement_untested = ~year_pair_can_be_checked
check_year_agreement = summarise_check(
    "CHK14", "Reported and discovery years agree", len(raw),
    year_agreement_flag.sum(), year_agreement_untested.sum()
)
evidence_year_agreement = raw.loc[
    year_agreement_flag | year_agreement_untested,
    ["fire_name", "fire_year", "discovery_date"]
].copy()
evidence_year_agreement["Interpreted reported year"] = reported_years.loc[evidence_year_agreement.index]
evidence_year_agreement["Discovery year"] = discovery_years.loc[evidence_year_agreement.index]
evidence_year_agreement["Record outcome"] = "Flagged"
evidence_year_agreement.loc[year_agreement_untested.loc[evidence_year_agreement.index], "Record outcome"] = "Untested"
display(pd.DataFrame([check_year_agreement]))
display(evidence_year_agreement.head(5))
save_check(
    check_year_agreement, evidence_year_agreement,
    finding="Reported-year agreement with discovery year has disagreements or unavailable comparisons.",
    effect="Q1 can assign a record to different annual groups depending on the year field used.",
    next_action="Review the original field definitions and affected source records before adopting a year field or correcting a value."
)


**Read the result:** Evidence shows both year values for each concern. A disagreement establishes that the values differ; it does not identify which extracted value should be corrected.



**Pause and explain:** Why is a record with reported year 1998.5 Untested for year agreement?

<details>
<summary>Suggested answer — open after discussing</summary>

Its reported year failed the whole-year prerequisite. That finding remains Flagged under CHK12.

</details>


## Block 5 · CHK13 · Compare the order of usable dates

**Objective:** Find reversed date pairs and measure how many pairs could not be compared.

**Expectation:** Containment is on or after discovery. Same-day containment meets the rule; a missing or unreadable endpoint prevents comparison.

**Where it comes from:** The chronology needed for discovery-to-containment duration, Q3.

**Objects used here:** `date_values` is the saved interpreted-date table loaded in Block 1. This block introduces both endpoint Series and creates every date-order result locally.

**Method reference:** [notna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.notna.html) · [fillna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.fillna.html) · [loc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html) · [copy()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html)


In [ ]:
discovery_for_order = date_values["discovery_date_value"]
containment_for_order = date_values["contained_date_value"]
date_pair_can_be_checked = discovery_for_order.notna() & containment_for_order.notna()
# < flags only containment BEFORE discovery. Equal dates remain unflagged.
containment_is_earlier = (containment_for_order < discovery_for_order).fillna(False)
date_order_flag = date_pair_can_be_checked & containment_is_earlier
date_order_untested = ~date_pair_can_be_checked
check_date_order = summarise_check(
    "CHK13", "Containment is not before discovery", len(raw),
    date_order_flag.sum(), date_order_untested.sum()
)
evidence_date_order = raw.loc[
    date_order_flag | date_order_untested,
    ["fire_name", "discovery_date", "contained_date"]
].copy()
evidence_date_order["Interpreted discovery"] = discovery_for_order.loc[evidence_date_order.index]
evidence_date_order["Interpreted containment"] = containment_for_order.loc[evidence_date_order.index]
evidence_date_order["Record outcome"] = "Flagged"
evidence_date_order.loc[date_order_untested.loc[evidence_date_order.index], "Record outcome"] = "Untested"
display(pd.DataFrame([check_date_order]))
display(evidence_date_order.head(5))
save_check(
    check_date_order, evidence_date_order,
    finding="Date order or the availability of a usable date pair requires attention.",
    effect="Q3 needs usable endpoints in the stated order. Untested pairs cannot supply a duration under this rule.",
    next_action="Investigate reversed pairs and unavailable endpoints separately. Define the duration population using both readiness and order."
)


**Read the result:** The summary can be Passed while still recording Untested pairs. That means the checked pairs met the rule; unavailable pairs remain a limitation. The helper therefore records an issue when Untested is positive, even if Flagged is zero.



**Pause and explain:** Why is zero reversed dates compatible with incomplete duration coverage?

<details>
<summary>Suggested answer — open after discussing</summary>

The reversal count concerns only usable pairs. Missing or unreadable endpoints could not enter it.

</details>


## Block 6 · Decide what to investigate before cleaning

**Objective:** Explain a proposed action that follows from the evidence.

| Finding | A justified proposed action | What still needs establishing |
| --- | --- | --- |
| Readable negative area | Review source units, recording and extraction. | The correct measurement or a documented exclusion decision. |
| Reversed date pair | Compare the original date records and interpretation. | Which endpoint or interpretation should change, if any. |
| Year disagreement | Review source definitions and the original record. | Which year field the analysis should use and whether a value needs correction. |
| Missing date endpoint | Seek recovery or define a duration subset. | The coverage and limits of that subset. |

**Inspection and cleaning:** Inspection records the original outcome and a proposed action.
Cleaning later applies an agreed treatment and reruns the same relevant check. Keep the original
inspection record so that before-and-after results can be explained. Writing a proposed action
does not resolve the issue or change a Flagged result to Passed.

**If your practical requires severity:** Base it on the planned question, not automatically on
a percentage. For Q3, unavailable dates may have a high effect because they block the calculation.
For Q2, outside spaces in a display name may have a smaller effect. State the reason. This study
guide does not invent automatic severity scores or replace your assessment instructions.


## Block 7 · Bring meaning and coverage together

**Student task:** Write a conclusion for Q3 using your displayed Checked, Flagged and Untested
counts. Then explain why the year agreement evidence matters to Q1.

**Suggested structure:** “For [question], [count] records could be checked. [count] need review
under [rule], and [count] could not be checked because [reason]. Before analysis, we need to
[specific investigation or population decision].”

**Do not add counts across checks:** CHK02, CHK10 and CHK13 can concern the same missing
containment dates for different reasons. Their combined counts are not distinct-record counts.

**Saved outputs:** CHK11–CHK14 now have their check, issue and complete evidence files.
Their numbering remains fixed even though CHK12 and CHK14 were taught together to keep
the year prerequisite close to its use. Chapter 5 will sort the references and assemble the report.
